In [ ]:
import mne #eeg analysis
import numpy as np # dataframes and staff
import matplotlib.pyplot as plt # plots
import matplotlib # plots
import PyQt5 # how plots open in you PC
from pathlib import Path # path to files
from mne.channels import find_ch_adjacency # connection btw electrodes
from functools import partial # statistics
import scipy.stats #statistics
import pickle # way to save data
from scipy import stats # stats
import statistics # stats
import mne_qt_browser
from mne.datasets import somato
import pathlib
import pandas as pd
from scipy.stats import wilcoxon
import mne.viz
import os
#from mne import mne_Loreta_plot

#the following magic line will make all of your figures to be of the chosen size (in inches)
%matplotlib qt5

mne.viz.set_browser_backend('qt')

In [ ]:
# для новых данных

path_to_fif_saves_offset_new = Path('/Users/alexmedvedev/Desktop/Vishka/new/fif_saves_offset')
path_to_fif_saves_tfr_offset_new = Path('/Users/alexmedvedev/Desktop/Vishka/new/fif_saves_offset_tfr')
path_to_fif_saves_onset_new = Path('/Users/alexmedvedev/Desktop/Vishka/new/fif_saves_onset')
path_to_fif_saves_tfr_onset_new = Path('/Users/alexmedvedev/Desktop/Vishka/new/fif_saves_onset_tfr')

In [ ]:
def unify_annotation_events_batch(data_directory, annotation_mapping, output_suffix="_unified"):
    """
    Process multiple FIF files to unify annotation-based events
    
    Parameters:
    -----------
    data_directory : str
        Directory containing the FIF files
    annotation_mapping : dict
        Dictionary mapping old annotation descriptions to new ones
        e.g., {'Stimulus/S 51': 'Stimulus/S  5'}  # merges S51 into S5
    """
    
    fif_files = list(Path(data_directory).glob("*.fif"))
    print(f"Processing {len(fif_files)} files")
    print(f"Annotation mapping: {annotation_mapping}")
    
    for i, fif_file in enumerate(fif_files, 1):
        print(f"\n[{i}/{len(fif_files)}] {fif_file.name}")
        
        # Load raw file
        raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
        
        if raw.annotations is None or len(raw.annotations) == 0:
            print("  No annotations found")
            continue
        
        # Show original annotations
        original_descriptions = list(set(raw.annotations.description))
        print(f"  Original: {original_descriptions}")
        
        # Create modified annotations
        annotations = raw.annotations.copy()
        changes_made = {}
        
        for old_desc, new_desc in annotation_mapping.items():
            mask = annotations.description == old_desc
            count_changed = np.sum(mask)
            if count_changed > 0:
                annotations.description[mask] = new_desc
                changes_made[old_desc] = (new_desc, count_changed)
                print(f"    '{old_desc}' -> '{new_desc}': {count_changed} events")
        
        # Show final annotations
        final_descriptions = list(set(annotations.description))
        print(f"  Final: {final_descriptions}")
        
        # Apply unified annotations to raw file
        raw_unified = raw.copy()
        raw_unified.set_annotations(annotations)
        
        # Save unified file
        output_file = fif_file.parent / f"{fif_file.name}"
        raw_unified.save(output_file, overwrite=True, verbose=False)
        print(f"  Saved: {output_file.name}")

Ids for new data:

{'New Segment/': 1,  
 'Noise': 2,  
 'Stimulus/S  5': 3,  
 'Stimulus/S  7': 4,  
 'Stimulus/S  9': 5,  
 'Stimulus/S 11': 6,  
 'Stimulus/S 15': 7,  
 'Stimulus/S 25': 8,  
 'Stimulus/S 27': 9,  
 'Stimulus/S 29': 10,  
 'Stimulus/S 35': 11,  
 'Stimulus/S 51': 12,  
 'Stimulus/S 63': 13,  
 'Stimulus/S 71': 14,  
 'Stimulus/S 91': 15,  
 'Stimulus/S111': 16,  
 'Stimulus/S211': 17,  
 'Stimulus/S251': 18}  

In [ ]:

# Usage
annotation_mapping = {
    'Stimulus/S 51': 'Stimulus/S  5', # rename the first one into the second one
    'Stimulus/S 71': 'Stimulus/S  7',
    'Stimulus/S 91': 'Stimulus/S  9',
    'Stimulus/S111': 'Stimulus/S 11',
    'Stimulus/S251': 'Stimulus/S 25',
    'Stimulus/S 15': 'Stimulus/S 27',
    'Stimulus/S 35': 'Stimulus/S 29',
    'Stimulus/S 63': 'Stimulus/S211',
}

In [ ]:
unify_annotation_events_batch(path_to_fif_saves_tfr_onset_new, annotation_mapping)